# BBEATSx Phase 3: Decomposition Fidelity & Exogenous Interpretability — DGP 5

This notebook validates BBEATSx on **DGP 5: Linear Trend + Seasonality + Linear Exogenous Covariate + Homoscedastic Noise**.
It measures point/probabilistic forecast calibration and demonstrates covariate interpretability via split importance and Partial Dependence Plots (PDP).

## Google Colab Setup

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Setting up workspace...")
    import os
    if not os.path.abspath(".").endswith("simulations"):
        if not os.path.exists("BBEATSx"):
            !git clone https://github.com/hugogobato/BBEATSx.git
        !pip install ./BBEATSx
        %cd BBEATSx/simulations
    else:
        print("Already in simulations directory.")

import sys
import os
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath(".."))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bbeatsx import BBEATSx, make_config
import utils

## Simulation Configuration

In [ ]:
NUM_SIMULATIONS = 100  # Set to 100 for final results on Colab
HORIZON = 20
LEVEL = 0.90
print(f"Running {NUM_SIMULATIONS} trials for DGP 5...")

## Run Simulations

In [ ]:
dgp5_forecasts = []
dgp5_components = []

for i in range(NUM_SIMULATIONS):
    t, y, exog, true_comps = utils.generate_dgp5(n=200, seed=i)
    n_train = len(y) - HORIZON
    y_train, y_test = y[:n_train], y[n_train:]
    
    exog_train = {k: v[:n_train] for k, v in exog.items()}
    exog_test = {k: v[n_train:] for k, v in exog.items()}
    
    cfg = make_config(
        periods=[(12, 3)],
        lags=(1,),
        exog=["x"],
        trend="spline",
        errors="homo",
        num_gfr=10,
        num_burnin=150,
        num_mcmc=300,
        seed=i
    )
    
    model = BBEATSx(cfg).fit(y_train, exog=exog_train)
    fc = model.forecast(HORIZON, exog_future=exog_test)
    
    dgp5_forecasts.append(utils.evaluate_forecast(y_test, fc, y_train, level=LEVEL))
    
    s = model.sampler_
    mean, std = s.y_mean_, s.y_std_
    in_sample_draws = s.in_sample_components()
    pred_in_sample = {
        "trend": mean + std * in_sample_draws["trend"],
        "seasonal": std * in_sample_draws["seasonal"],
        "generic": std * in_sample_draws["generic"]
    }
    comp_metrics_in = utils.evaluate_component_fidelity(
        true_comps, pred_in_sample, offset=s.fs.row_offset, level=LEVEL
    )
    
    pred_out_sample = {
        "trend": fc.components["trend"],
        "seasonal": fc.components["seasonal"],
        "generic": fc.components["generic"]
    }
    true_out_comps = {k: v[n_train:] for k, v in true_comps.items()}
    comp_metrics_out = utils.evaluate_component_fidelity(
        true_out_comps, pred_out_sample, offset=0, level=LEVEL
    )
    
    dgp5_components.append({
        "in_sample": comp_metrics_in,
        "out_sample": comp_metrics_out
    })
    
    if (i + 1) % 10 == 0:
        print(f"  Completed trial {i + 1}/{NUM_SIMULATIONS}")

## Results Summary

In [ ]:
df_fc5 = pd.DataFrame(dgp5_forecasts)
print("\n--- DGP 5 Forecasting Calibration Battery ---")
print(df_fc5.mean().to_frame("Mean Metric"))

comp_in_rmse = {k: np.mean([t["in_sample"][k]["rmse"] for t in dgp5_components]) for k in ["trend", "seasonal", "generic"]}
comp_in_cov = {k: np.mean([t["in_sample"][k]["coverage"] for t in dgp5_components]) for k in ["trend", "seasonal", "generic"]}
comp_out_rmse = {k: np.mean([t["out_sample"][k]["rmse"] for t in dgp5_components]) for k in ["trend", "seasonal", "generic"]}
comp_out_cov = {k: np.mean([t["out_sample"][k]["coverage"] for t in dgp5_components]) for k in ["trend", "seasonal", "generic"]}

df_comp5 = pd.DataFrame({
    "In-Sample RMSE": comp_in_rmse,
    "In-Sample Coverage (90%)": comp_in_cov,
    "Out-of-Sample RMSE": comp_out_rmse,
    "Out-of-Sample Coverage (90%)": comp_out_cov
})
print("\n--- DGP 5 Component Fidelity (Plan §3.2) ---")
print(df_comp5)

## Covariate Interpretability: Variable Importance & Partial Dependence Plot (PDP)

We check the variable split importance in the generic block, and verify if the Partial Dependence Plot successfully recovers the true linear effect ($2.0 \cdot x$).

In [ ]:
# 1. Split-frequency importance
imp = model.split_importance("generic")
print("\nGeneric block split importance:")
for name, m, p in zip(imp.names, imp.mean, imp.inclusion_prob):
    print(f"  {name:8s}  mean splits={m:6.2f}  P(used)={p:.2f}")

# 2. Partial Dependence Plot for 'x'
names_ge = list(model.sampler_.generic_block.feature_names)
x_idx = names_ge.index("x")

grid = np.linspace(-2, 2, 50)
pdp = model.partial_dependence(x_idx, grid, block="generic", level=LEVEL)

plt.figure(figsize=(7, 4))
plt.plot(pdp.x, pdp.mean, color='blue', label='BBEATSx Recovered PDP (mean)')
plt.fill_between(pdp.x, pdp.lower, pdp.upper, color='blue', alpha=0.15, label='90% Credible Band')
plt.plot(grid, 2.0 * grid, color='red', linestyle='--', label='True Linear Effect (2.0 * x)')
plt.xlabel("Covariate x")
plt.ylabel("Effect on Target")
plt.title("DGP 5: Partial Dependence Plot (Linear Exogenous Covariate Recovery)")
plt.legend()
plt.grid(True)
plt.show()